# Exercise 3.4 
This exercise is for you to implement the car engine management system that you created in Activity 1.1 and answer the query: given the led and temperature gauge are working and the led lights,"what is the probaiblity that the temperature of the engine is too high?".

The code below provides a solution but you should initially attempt this from scratch.

In your implementation assume:

- there are just two possible actual and measured temperatures of the collant, normal and high; the probability that the gauge gives the correct temperature is x when it is working, but y when it is faulty. 

- the led works correctly unless it is faulty, in which case it never sounds.



In [128]:
from pgmpy.models import BayesianModel
import matplotlib.pyplot as plt
from pgmpy.factors.discrete.CPD import TabularCPD
EMS = BayesianModel()

As before we can use a shortcut to create nodes and edges at the same time

In [129]:
EMS.add_edge(u='T',v='FG')
EMS.add_edge(u='FG',v='GT')
EMS.add_edge(u='T',v='GT')
EMS.add_edge(u='GT',v='L')
EMS.add_edge(u='FL',v='L')

In [130]:
EMS.edges()

OutEdgeView([('T', 'FG'), ('T', 'GT'), ('FG', 'GT'), ('GT', 'L'), ('FL', 'L')])

Let's set the probability that the gauge gives the correct temperature is 0.999 when it is working, but 0.1 when it is faulty. 
I am using the prior P(T=High) = 0.05 and P(FG|T)={0.01,0.04}

In [159]:
x= 0.999
y= 0.1
cpd_GT = TabularCPD('GT', 2, [[x,y,1-x,1-y],
                                      [1-x, 1-y, x, y]],
                   evidence=['T','FG'],
                   evidence_card=[2,2])
cpd_L = TabularCPD('L', 2, [[1,1,0,1],
                                      [0, 0, 1, 0]],
                   evidence=['GT','FL'],
                   evidence_card=[2,2])
cpd_T = TabularCPD('T',2,[[0.95],[0.05]])
cpd_FG = TabularCPD('FG',2,[[0.99,0.96],[0.01,0.04]],evidence=['T'],evidence_card=[2])
cpd_FL = TabularCPD('FL',2,[[0.99],[0.01]])



EMS.add_cpds(cpd_GT, cpd_L,cpd_T,cpd_FG,cpd_FL)

In [160]:
# Checking if the cpds are valid for the model
EMS.check_model()

True

We can display all the CPDs if we wish to check them

In [161]:
print(cpd_GT)

+-------+-----------------------+-------+-----------------------+-------+
| T     | T(0)                  | T(0)  | T(1)                  | T(1)  |
+-------+-----------------------+-------+-----------------------+-------+
| FG    | FG(0)                 | FG(1) | FG(0)                 | FG(1) |
+-------+-----------------------+-------+-----------------------+-------+
| GT(0) | 0.999                 | 0.1   | 0.0010000000000000009 | 0.9   |
+-------+-----------------------+-------+-----------------------+-------+
| GT(1) | 0.0010000000000000009 | 0.9   | 0.999                 | 0.1   |
+-------+-----------------------+-------+-----------------------+-------+


In [162]:
print(cpd_L)

+------+-------+-------+-------+-------+
| GT   | GT(0) | GT(0) | GT(1) | GT(1) |
+------+-------+-------+-------+-------+
| FL   | FL(0) | FL(1) | FL(0) | FL(1) |
+------+-------+-------+-------+-------+
| L(0) | 1.0   | 1.0   | 0.0   | 1.0   |
+------+-------+-------+-------+-------+
| L(1) | 0.0   | 0.0   | 1.0   | 0.0   |
+------+-------+-------+-------+-------+


In [163]:
print(cpd_T)

+------+------+
| T(0) | 0.95 |
+------+------+
| T(1) | 0.05 |
+------+------+


In [164]:
print(cpd_FG)

+-------+------+------+
| T     | T(0) | T(1) |
+-------+------+------+
| FG(0) | 0.99 | 0.96 |
+-------+------+------+
| FG(1) | 0.01 | 0.04 |
+-------+------+------+


In [165]:
print(cpd_FL)

+-------+------+
| FL(0) | 0.99 |
+-------+------+
| FL(1) | 0.01 |
+-------+------+


In [166]:
# Getting all the independencies in the network.
EMS.get_independencies()

(T ⟂ FL)
(T ⟂ FL, L | GT)
(T ⟂ FL | FG)
(T ⟂ FL, L | GT, FG)
(T ⟂ L | GT, FL)
(T ⟂ FL | GT, L)
(T ⟂ L | GT, FG, FL)
(T ⟂ FL | GT, FG, L)
(FL ⟂ T, FG, GT)
(FL ⟂ GT, FG | T)
(FL ⟂ T, GT | FG)
(FL ⟂ T, FG | GT)
(FL ⟂ GT | T, FG)
(FL ⟂ FG | T, GT)
(FL ⟂ T | GT, FG)
(FL ⟂ T, FG | GT, L)
(FL ⟂ FG | T, GT, L)
(FL ⟂ T | GT, FG, L)
(GT ⟂ FL)
(GT ⟂ FL | T)
(GT ⟂ FL | FG)
(GT ⟂ FL | T, FG)
(FG ⟂ FL)
(FG ⟂ FL | T)
(FG ⟂ FL, L | GT)
(FG ⟂ FL, L | T, GT)
(FG ⟂ L | GT, FL)
(FG ⟂ FL | GT, L)
(FG ⟂ L | T, GT, FL)
(FG ⟂ FL | T, GT, L)
(L ⟂ T, FG | GT)
(L ⟂ FG | T, GT)
(L ⟂ T | GT, FG)
(L ⟂ T, FG | GT, FL)
(L ⟂ FG | T, GT, FL)
(L ⟂ T | GT, FG, FL)

Our query is, "what is the probability temperature of engine too high P(T)?"
Given the evidence that led is on (L = 1), and that it is not faulty (FL = 0). The Gauge is also working (FG = 0)


In [167]:
from pgmpy.inference import VariableElimination

infer = VariableElimination(EMS)
posterior_p = infer.query(['T'], evidence={'FG': 0,'FL' : 0, 'L': 1}) 
print(posterior_p)


Finding Elimination Order: :   0%|                            | 0/1 [00:00<?, ?it/s]


  0%|                                                         | 0/1 [00:00<?, ?it/s]


Eliminating: GT: 100%|███████████████████████████████| 1/1 [00:00<00:00, 247.52it/s]

+------+----------+
| T    |   phi(T) |
+======+==========+
| T(0) |   0.0192 |
+------+----------+
| T(1) |   0.9808 |
+------+----------+
